# Show the Barotropic Flow in the 79NG Fjord

Notebook by Markus Reinert (IOW, 2024–2026, https://orcid.org/0000-0002-3761-8029).

In [ ]:
import gsw
import numpy as np
import xarray as xr
import cmocean.cm as cmo
import matplotlib.pyplot as plt

from tools.visualization import cm

## Load the model result

In [ ]:
ds = xr.open_dataset("output/out_mean_compressed.nc")

# Convert single time value from coordinate to attribute
assert ds.time.size == 1
time_value = ds.time.data[0]
ds = ds.squeeze(dim="time", drop=True)
ds.attrs["time"] = time_value

# Compute the mask
ds["mask"] = ds.bathymetry.notnull()
ds.mask.attrs = {"long_name": "ocean mask"}

# Check grid spacing
dlat = 1 / 240
dlon = 3 / 120
assert np.allclose(np.diff(ds.latc), dlat)
assert np.allclose(np.diff(ds.lonc), dlon)

# Use this aspect ratio in lat–lon plots for squared grid cells; since dx and dy
# are approximately equal, this gives an approximate equal-area map.
grille_carree = dlon / dlat

print(f"Total number of grid points: {ds.bathymetry.size:_d}")
print(f"Number of ocean grid points: {ds.bathymetry.count():_d}")
print(f"Ratio of ocean to all cells: {ds.bathymetry.count() / ds.bathymetry.size * 100:.0f} %")

ds

## Compute the data

In [ ]:
mag = np.sqrt(ds.U**2 + ds.V**2).where(ds.mask)
mag.attrs = {"long_name": "transport", "units": "m$^2$ s$^{-1}$"}
mag = mag.sel(lonc=slice(-19 + dlon/2), latc=slice(79.25 - dlat/2, 79.85 + dlat/2))

n = 5
Q = ds.isel(lonc=slice(0, None, n), latc=slice(0, None, n))
Q = Q.where(Q.mask)
print(f"Note: only 1 out of every {n*n} arrows shown.")

## Create the figure

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(12*cm, 11.5*cm), dpi=300)
ax.set_facecolor("0.8")

print(f"Maximum water column thickness: {ds.D.max():.1f} m")
cs = ds.D.plot.contour(ax=ax, levels=np.arange(100, 800, 100), linewidths=0.8, colors="gray")

print(f"Maximum transport magnitude: {mag.max():.3f} {mag.units}")
im = mag.plot(ax=ax, vmin=0, vmax=15, cmap=cmo.amp, add_colorbar=False)
cbar = fig.colorbar(
    im, ax.inset_axes((-22.5, 79.79, 1.7, 0.02), transform=ax.transData),
    orientation="horizontal", ticks=[0, 5, 10, 15],
)
cbar.set_label(f"vertically integrated velocity\nmagnitude in {mag.units}")

Q_plot = ax.quiver(Q.lonc, Q.latc, Q.U, Q.V, scale=250, color="#f2f200", zorder=cs.zorder+1, width=0.004)
qk = ax.quiverkey(Q_plot, -21.65, 79.65, -15, f"15 {mag.units}", coordinates="data", labelsep=0.05, fontproperties={"size": "small"})

color = "darkblue"
ds.glIceD.plot.contour(ax=ax, levels=[0], colors=color, linewidths=1.2, zorder=cs.zorder+2)
ax.annotate(
    "calving\nfront", (-19.64, 79.41), (-19.9, 79.32),
    arrowprops={"arrowstyle": "->", "color": color, "shrinkA": 0, "shrinkB": 1},
    color=color, ha="center",
)

ax.set_title("Barotropic flow\non contours of water column thickness")
ax.set(xlabel="", ylabel="", xlim=(-22.7, -19), ylim=(79.25, 79.85), aspect=grille_carree)
xticks = [-22, -21, -20, -19]
ax.set_xticks(xticks, [f"{-x:.0f}°W" for x in xticks])
ax.yaxis.set_major_formatter(lambda x, pos: f"{round(x, 10)}°N")

# Show a 10 km-scale
# The length of the scale is accurate at the shown position.
# Due to the map projection, the length of the scale differs slightly
# if it is moved to a different location, see the printed text.
# Choose the center point position and the length of the scale
lat_scale = 79.8
lon_scale = -19.5
dx_scale = 10e3
# Compute the length of the scale in degrees longitude
d_lon_scale = dx_scale / gsw.distance([0, 1], [lat_scale, lat_scale])[0]
print(f"Length of the shown {dx_scale / 1e3:.0f} km-scale: {d_lon_scale:.3f}° longitude")
# Show and label the scale
lw = 1
ax.plot([lon_scale - d_lon_scale / 2, lon_scale + d_lon_scale / 2], [lat_scale, lat_scale], "k|-", lw=lw, mew=lw, solid_capstyle="butt")
ax.text(lon_scale, lat_scale - 0.02, f"{dx_scale / 1e3:.0f} km", va="top", ha="center")
# Compute the length of the scale if it is moved or rotated on the map
print(f"Length of the {dx_scale / 1e3:.0f} km-scale in the South: {gsw.distance([lon_scale, lon_scale + d_lon_scale], [79.25, 79.25])[0] / 1e3:.3f} km")
print(f"Length of the {dx_scale / 1e3:.0f} km-scale in the North: {gsw.distance([lon_scale, lon_scale + d_lon_scale], [79.85, 79.85])[0] / 1e3:.3f} km")
print(f"Length of the {dx_scale / 1e3:.0f} km-scale turned 90°:   {gsw.distance([0, 0], [lat_scale, lat_scale + d_lon_scale / grille_carree])[0] / 1e3:.3f} km")

fig.savefig("figures/fig_2_barotropic_flow.png", dpi=600)